# Anomaly Detection on a Multiple Time Series using Snowflake's ML Function

## The Data

Definition: `(as_of_date, key_1, key_2, key_3, value)`   

A Multiple time-series dataset where each row is uniquely identiable by the combination of `(key_1,key_2,key_3)`.

Ref: sample.csv & ts_data.csv

In [1]:
import pandas as pd 
import plotly.express as px

df = pd.read_csv('./data/ts_data.csv')
df.head()

,as_of_date,key_1,key_2,key_3,value
0,2023-01-01,key_1_a,key_2_a,key_3_a,7833.373265
1,2023-01-02,key_1_a,key_2_a,key_3_a,8002.271356
2,2023-01-03,key_1_a,key_2_a,key_3_a,8009.132334
3,2023-01-04,key_1_a,key_2_a,key_3_a,8187.491965
4,2023-01-05,key_1_a,key_2_a,key_3_a,7589.500133


Looking at one of these many series: `(key_1_a, key_2_a, key_3_a)`

Things to notice:   
    - Data duration (2 + 1 years)   
    - Anomalies dumped in the last year.   
    - Seasonality

In [3]:
target_series = df[(df['key_1'] == 'key_1_a') & 
                   (df['key_2'] == 'key_2_a') & 
                   (df['key_3'] == 'key_3_a')]

fig = px.line(target_series, x='as_of_date', y='value', 
              title='Our time series data (key_1_a, key_2_a, key_3_a)')

fig.write_html(f'./plots/series1-plot.html')

Another series: key_1_b, key_2_a, key_3_b

In [4]:
target_series = df[(df['key_1'] == 'key_1_b') & 
                   (df['key_2'] == 'key_2_a') & 
                   (df['key_3'] == 'key_3_b')]

fig = px.line(target_series, x='as_of_date', y='value', 
              title='Our time series data (key_1_b, key_2_a, key_3_b)')

fig.write_html(f'./plots/series2-plot.html')

## Snowflake's solution

First, we will split our 3 years data into 2+1 years.   

1. Training data view with first 2 years of 'normal data'.  

    ```snowflake
    create or replace view ts_series_training as
    select 
        ["key_1", "key_2", "key_3"] as row_key, 
        to_timestamp("as_of_date") as as_of_date, 
        "value" as value
    from ts_series
    where to_timestamp("as_of_date") < to_timestamp('2025-01-01');
    ```

2. Testing data view with last year of 'anomalous data'.

    ```snowflake
    create or replace view ts_series_test as
    select 
        ["key_1", "key_2", "key_3"] as row_key, 
        to_timestamp("as_of_date") as as_of_date, 
        "value" as value
    from ts_series
    where to_timestamp("as_of_date") > to_timestamp('2025-01-01');
    ```

Next, we will train snowflake's ML function of anomaly detection:

Params: 
1. Input data: The input data table. Here, the training view in the table method.
2. Series colname: The key that defines each series.
3. Timeseries colname: The as of date.
4. Target colname: Column to monitor, here value.
5. Label colname: is-anomaly column; only if we have clear mappings of the historical data.

```snowflake
create or replace snowflake.ml.anomaly_detection holmes_beta (
    input_data => table(ts_series_training),
    series_colname => 'row_key',
    timestamp_colname => 'as_of_date',
    target_colname => 'value',
    label_colname => ''
);
```

Note: Will take around 18 minutes to train.

Next, we will try calling it on the testing view.

Params:
1. Input data: The testing data in the table method (if not table).
2. Series colname: The key defining each time series.
3. Timestamp colname: The as-of-date column.
4. Target colnameL: The target column to monitor value of.
5. Config object: Additional configuration. Here, we have set the prediction interval to 0.999 to keep the model extra strict.

```snowflake
    call holmes_beta!detect_anomalies(
    input_data => table(ts_series_test),
    series_colname => 'row_key',
    timestamp_colname => 'as_of_date',
    target_colname => 'value',
    config_object => {
       'prediction_interval': 0.999
    }
)
```

## Analysis of the output

I've stored the output in `data/result.csv` file.

In [5]:
df_result = pd.read_csv('./data/result.csv')
df_result.head()

,SERIES,TS,Y,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE
0,"[\n ""key_1_b"",\n ""key_2_a"",\n ""key_3_c""\n]",2025-07-07 00:00:00.000,7352.658754,7285.171227,6898.982089,7671.360365,False,0.717364,0.575028
1,"[\n ""key_1_b"",\n ""key_2_a"",\n ""key_3_c""\n]",2025-07-08 00:00:00.000,7388.109942,7228.740861,6842.551723,7614.929999,False,0.912753,1.357905
2,"[\n ""key_1_b"",\n ""key_2_a"",\n ""key_3_c""\n]",2025-07-09 00:00:00.000,6932.278078,6883.174113,6496.984975,7269.363251,False,0.662169,0.418391
3,"[\n ""key_1_b"",\n ""key_2_a"",\n ""key_3_c""\n]",2025-07-10 00:00:00.000,6813.516465,6669.620598,6283.431460,7055.809737,False,0.889913,1.226066
4,"[\n ""key_1_b"",\n ""key_2_a"",\n ""key_3_c""\n]",2025-07-11 00:00:00.000,6606.517137,6661.688486,6275.499347,7047.877624,False,0.319146,-0.470088


In [6]:
# Bit of cleaning

def clean_key(col):
    return col.str.replace(r'[\[\]",]', '', regex=True).str.strip()

split_keys = df_result['SERIES'].str.split('\n', expand=True)
df_result['key_1'] = clean_key(split_keys[1])
df_result['key_2'] = clean_key(split_keys[2])
df_result['key_3'] = clean_key(split_keys[3])

df_result['as_of_date'] = pd.to_datetime(df_result['TS']).dt.date
df_result['value'] = df_result['Y']
df_result = df_result.drop(columns=['SERIES', 'TS', 'Y'])

df_result.head()

,FORECAST,LOWER_BOUND,UPPER_BOUND,IS_ANOMALY,PERCENTILE,DISTANCE,key_1,key_2,key_3,as_of_date,value
0,7285.171227,6898.982089,7671.360365,False,0.717364,0.575028,key_1_b,key_2_a,key_3_c,2025-07-07,7352.658754
1,7228.740861,6842.551723,7614.929999,False,0.912753,1.357905,key_1_b,key_2_a,key_3_c,2025-07-08,7388.109942
2,6883.174113,6496.984975,7269.363251,False,0.662169,0.418391,key_1_b,key_2_a,key_3_c,2025-07-09,6932.278078
3,6669.620598,6283.431460,7055.809737,False,0.889913,1.226066,key_1_b,key_2_a,key_3_c,2025-07-10,6813.516465
4,6661.688486,6275.499347,7047.877624,False,0.319146,-0.470088,key_1_b,key_2_a,key_3_c,2025-07-11,6606.517137


In [7]:
from itertools import product
import plotly.graph_objects as go

allowed_key_1 = ['key_1_a', 'key_1_b', 'key_1_c']
allowed_key_2 = ['key_2_a', 'key_2_b', 'key_2_c']    
allowed_key_3 = ['key_3_a', 'key_3_b', 'key_3_c']    

dfs = []

for key1,key2,key3 in product(allowed_key_1, allowed_key_2, allowed_key_3):
    filtered_df = df_result[
        (df_result['key_1'] == key1) & 
        (df_result['key_2'] == key2) & 
        (df_result['key_3'] == key3)
    ]
    if not filtered_df.empty:
        dfs.append(filtered_df)
    
for idx, d in enumerate(dfs):
    if d.empty:
        continue
    
    d = d.sort_values('as_of_date')
    d['dist_upper'] = (d['value'] - d['UPPER_BOUND']).clip(lower=0)
    d['dist_lower'] = (d['value'] - d['LOWER_BOUND']).clip(upper=0)
    d['total_dist'] = d['dist_upper'] + d['dist_lower']

    k1, k2, k3 = d['key_1'].iloc[0], d['key_2'].iloc[0], d['key_3'].iloc[0]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=pd.concat([d['as_of_date'], d['as_of_date'][::-1]]),
        y=pd.concat([d['UPPER_BOUND'], d['LOWER_BOUND'][::-1]]),
        fill='toself',
        fillcolor='rgba(128, 128, 128, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        name='Safe Zone (PI)',
    ))

    fig.add_trace(go.Scatter(
        x=d['as_of_date'], 
        y=d['FORECAST'],
        name='Baseline Forecast',
        line=dict(color='rgba(255, 127, 14, 0.7)', width=1.5) 
    ))

    fig.add_trace(go.Scatter(
        x=d['as_of_date'], 
        y=d['value'],
        name='Actual Value',
        mode='lines',
        line=dict(color='#1f77b4', width=2),
        customdata=d[['UPPER_BOUND', 'LOWER_BOUND', 'total_dist']],
        hovertemplate=(
            "<b>Actual: %{y:,.2f}</b><br>" +
            "Upper: %{customdata[0]:,.2f}<br>" +
            "Lower: %{customdata[1]:,.2f}<br>" +
            "Dist from Bound: %{customdata[2]:,.2f}<extra></extra>"
        )
    ))

    anomalies = d[d['IS_ANOMALY'] == True]
    if not anomalies.empty:
        fig.add_trace(go.Scatter(
            x=anomalies['as_of_date'],
            y=anomalies['value'],
            name='Anomaly Detected',
            mode='markers',
            marker=dict(color='#d62728', size=5)
        ))

    fig.update_layout(
        title=f"<b>Dynamic Anomaly Detection</b><br><sup>{k1} | {k2} | {k3}</sup>",
        xaxis_title="Date",
        yaxis_title="Metric Value",
        template="plotly_white",
        width=1400,
        height=600,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray'),
        yaxis=dict(showgrid=True, gridcolor='lightgray')
    )

    fig.write_html(f'./plots/anomaly_plot_{idx}.html')